<a href="https://colab.research.google.com/github/joj-ojo/Desafios-tecnicos/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [11]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@desafio-tecnico-513817.iam.gserviceaccount.com


In [12]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1_6iS5pfCj7L5ps8D8hO03V-9DrqgPWOenQOyf-G9SXM"
NOME_TABELA = "pacientes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Posto_Saude | pacientes | https://docs.google.com/spreadsheets/d/1_6iS5pfCj7L5ps8D8hO03V-9DrqgPWOenQOyf-G9SXM


In [13]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["paciente_id", "nome_ficticio", "data_nascimento", "sexo", "bairro", "tipo_convenio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [14]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "paciente_id": fake.cpf(), #cria um cpf fake acionando função da bilbioteca faker
        "nome_ficticio": fake.name(), #cria um nome fake acionando função da bilbioteca faker
        "data_nascimento": fake.date_this_year().isoformat(),   # datas como texto
        "sexo": random.choice(["F", "M", "Outro"]), #seleciona uma das opções listadas aleatoriamente
        "bairro": fake.bairro(), #cria um  bairro fake acionando função da bilbioteca faker
        "tipo_convenio": random.choice(["SUS", "Particular", "Convenio"]) #seleciona uma das opções listadas aleatoriamente
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


In [16]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 41
Cabeçalho confere: True
[['paciente_id', 'nome_ficticio', 'data_nascimento', 'sexo', 'bairro', 'tipo_convenio'], ['1', 'Brenda Alves', '2026-08-17', 'N/B', 'Glória', 'SUS'], ['2', 'Davi Miguel Sousa', '2026-02-01', 'F', 'Vila Pinho', 'Particular']]


**PARE AQUI NA SESSÃO**

In [ ]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
import pandas as pd
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "pacientes": pd.read_csv("pacientes.csv"),
    "atendimento": pd.read_csv("atendimentos.csv"),
}
# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving pacientes.csv to pacientes (2).csv
Saving atendimentos.csv to atendimentos (2).csv
{'pacientes': 10500, 'atendimento': 42000} | total: 52500


In [24]:
# Aqui estamos ensinando ao Python o que a palavra 'enviar' significa
def enviar(sh, df, nome_aba):
    # Converte o DataFrame do Pandas em uma lista que o Google Sheets entende
    dados = [df.columns.values.tolist()] + df.values.tolist()

    # Cria a nova aba com o nome correto
    aba = sh.add_worksheet(title=nome_aba, rows=len(dados), cols=len(df.columns))

    # Joga os dados lá dentro
    aba.update(dados)

# E aqui ensinamos o que 'remover_aba_padrao' significa
def remover_aba_padrao(sh):
    try:
        aba_padrao = sh.worksheet("Página1")
    except:
        try:
            aba_padrao = sh.worksheet("Sheet1")
        except:
            return
    if not aba_padrao.acell('A1').value:
        sh.del_worksheet(aba_padrao)

# Abre a planilha pelo nome exato dela no seu Google Drive
sh = gc.open("Desafio_01_Posto_Saude")

# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)


In [21]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 52.500
A planilha confere com a base gerada.
